In [1]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


## Basic Word Count ##

The following Spark program will process a single text: Romeo & Juliet, which contained in the file `books\pg1513.txt`. In processing, a key-value pair to map each distinct word to its respective amount of occurences will be stored in the file `outputs\pg1513_basic_word_count.txt` in order of descending occurence.

In [2]:
from pyspark import SparkContext
from operator import add

sc = SparkContext(appName="WordCount") #Initialize spark
print(f'URL for SparkUI: {sc.uiWebUrl}')
text_file = sc.textFile("books\pg1513.txt")
word_counts = text_file.flatMap(lambda line: line.split(" ")) \
                        .map(lambda word: (word, 1)) \
                        .reduceByKey(add) \
                        .sortBy(lambda x: x[1], ascending=False)  #Sort in descending order
with open("outputs\pg1513_basic_word_count.txt", "w",encoding="utf-8") as file: #Collect data and save to test file
    for word_to_count in word_counts.collect():
        file.write(f"{word_to_count[0]}: {word_to_count[1]}\n")
sc.stop()  #Close Spark

URL for SparkUI: http://DESKTOP-F19QHQD:4040


## Extended Word Count ##

The following will extend the prior Spark program to process two texts: Romeo & Juliet and The Iliad, which are stored in `books\pg1513.txt` and `books\pg6130.txt`, respectively. In processing, each character will be lowercased to allow for case-insensitive analysis. Regular expressions will be utilized to remove all punction from the text. Finally, a list of common 'stop words' will be removed to allow for more comprehensive results.

After processing each text, a key-value pair mapping each word to its respective amount of occurences will be stored as a line in an output text file located in the `outputs` folder. Each file will be sorted decreasing with the most frequently occuring word first.

In [3]:
import re

sc = SparkContext(appName="WordCount")  #Initialize Spark
print(f'URL for SparkUI: {sc.uiWebUrl}')
#Define set of stop words
stop_words = set(["the", "and", "is", "in", "it", "of", "to", "a", "for", "on", "an", "as", "at"])

for text in ['pg6130.txt', 'pg1513.txt']:  #Process each text
    text_file = sc.textFile(f"books/{text}")

    #Function to remove all non-alphanumeric characters and convert to lowercase
    def clean(line):
        line = re.sub(r'[^\w\s]', '', line).lower() #Convert to lowercase and remove punctuation
        return line.split()

    # Count words after cleaning and filtering
    word_counts = text_file.flatMap(clean) \
        .map(lambda word: (word, 1)) \
        .filter(lambda x: x[0] not in stop_words) \
        .reduceByKey(add) \
        .sortBy(lambda x: x[1], ascending=False)  #Sort in descending order
    
    with open(f"outputs\{text[:6]}_extended_word_count.txt", "w",encoding="utf-8") as file: #Collect data and save to test file
        for word_to_count in word_counts.collect():
            file.write(f"{word_to_count[0]}: {word_to_count[1]}\n")
        file.close()

sc.stop()  #Close Spark

URL for SparkUI: http://DESKTOP-F19QHQD:4040
